# 02 模块分开配置

目标：把架构配置、分词器、模型加载参数和生成参数拆开，理解部署时每类配置控制什么。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


## 1. 架构配置

`AutoConfig` 保存层数、hidden size、attention heads、模型类型等结构信息。


In [ ]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
)


config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("model_type:", config.model_type)
print("hidden_size:", getattr(config, "hidden_size", "unknown"))
print("num_hidden_layers:", getattr(config, "num_hidden_layers", "unknown"))
print("num_attention_heads:", getattr(config, "num_attention_heads", "unknown"))
print("num_key_value_heads:", getattr(config, "num_key_value_heads", "unknown"))


## 2. 分词器和模型加载

部署时常见的关键项是 `device_map`、`torch_dtype`、`pad_token` 和 `eos_token`。


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    padding_side="left",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    config=config,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)


## 3. 生成配置

`GenerationConfig` 管 max_new_tokens、temperature、top_p、重复惩罚和停止 token。


In [ ]:
generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
generation_config.max_new_tokens = 120
generation_config.do_sample = True
generation_config.temperature = 0.7
generation_config.top_p = 0.9
generation_config.repetition_penalty = 1.05
generation_config.pad_token_id = tokenizer.pad_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

generation_config


## 4. 运行生成

这里开始能清楚看到：prompt 渲染、tokenize、generate、decode 是四个独立步骤。


In [ ]:
messages = [
    {"role": "system", "content": "你是一个面试辅导老师。"},
    {"role": "user", "content": "解释 prefill 和 decode 的区别。"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    generation_config=generation_config,
)

new_token_ids = outputs[0][inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(new_token_ids, skip_special_tokens=True)

print(answer)
